# PennyLane Apple Silicon scaling

Sweep statevector widths around MettleQ's automatic GPU crossover and verify complete-state parity.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the scalable workload

default.qubit runs on the Apple CPU. MettleQ keeps small work on CPU and moves sufficiently large exact states to MLX on the integrated GPU.

In [2]:
import statistics

def make_qnode(device, width):
    @qml.qnode(device)
    def circuit():
        for layer in range(3):
            for wire in range(width):
                qml.RY(0.01 * (layer + 1) * (wire + 1), wires=wire)
            for wire in range(layer % 2, width - 1, 2):
                qml.CNOT(wires=[wire, wire + 1])
        return qml.state()
    return circuit

rows = []

## 2. Run the same workload through both simulators

For each width we warm up both paths twice, take five measured runs, compare the complete state, record MettleQ's selected device, and print the result as a per-width table. A ratio above 1.0 means MettleQ was faster; below 1.0 means the SDK reference was faster.

In [3]:
for width in (12, 14, 16, 18, 20):
    reference_qnode = make_qnode(qml.device("default.qubit", wires=width), width)
    reference, reference_ms, _ = benchmark(reference_qnode, warmups=2, repeats=5)
    mettleq_device = MettleQDevice(wires=width, method="statevector", device="auto")
    mettleq_qnode = make_qnode(mettleq_device, width)
    candidate, mettleq_ms, _ = benchmark(mettleq_qnode, warmups=2, repeats=5)
    method, device = pennylane_selection(mettleq_device)
    rows.append({
        "width": width,
        "reference_ms": reference_ms,
        "mettleq_ms": mettleq_ms,
        "error": phase_aligned_statevector_error(reference, candidate),
        "method": method,
        "device": device,
    })

print_scaling_table(rows)
largest_width_speedup = rows[-1]["reference_ms"] / rows[-1]["mettleq_ms"]

qubits | reference ms | MettleQ ms | ref/MettleQ | path | max error
------ | ------------ | ---------- | ----------- | ---- | ---------
    12 |        3.561 |      4.415 |      0.807x | statevector/cpu | 1.31e-07
    14 |        6.699 |      7.052 |      0.950x | statevector/cpu | 1.67e-07
    16 |       14.106 |      5.738 |      2.458x | statevector/gpu | 6.96e-08
    18 |       49.707 |      6.760 |      7.353x | statevector/gpu | 1.18e-07
    20 |      380.684 |      7.529 |     50.565x | statevector/gpu | 1.61e-07


## 3. Enforce correctness and publish the evidence

Each full state is phase-aligned against default.qubit before reporting a speed ratio.

In [4]:
tutorial_result = emit_result(
    notebook="pennylane/13_apple_gpu_scaling.ipynb",
    framework="pennylane",
    reference_ms=statistics.median(row["reference_ms"] for row in rows),
    mettleq_ms=statistics.median(row["mettleq_ms"] for row in rows),
    check="per-width statevector atol=3e-6, policy-selected GPU, and 20q speedup >=1.5x",
    passed=(
        all(row["error"] <= 3e-6 for row in rows)
        and rows[-1]["device"] == "gpu"
        and largest_width_speedup >= 1.5
    ),
    exact_match=all(row["error"] == 0.0 for row in rows),
    selected_method=rows[-1]["method"],
    selected_device=rows[-1]["device"],
    metrics={"widths": rows},
    notes="The aggregate medians summarize different widths. The printed per-width table is the performance evidence; ratios above 1 mean MettleQ was faster.",
)


Comparison summary
------------------
Correctness contract: PASS — per-width statevector atol=3e-6, policy-selected GPU, and 20q speedup >=1.5x
SDK reference median: 14.106 ms
MettleQ median:       6.760 ms
Timing interpretation: MettleQ was 2.087x faster in this run.
MettleQ selected: statevector / gpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)
Note: The aggregate medians summarize different widths. The printed per-width table is the performance evidence; ratios above 1 mean MettleQ was faster.

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "per-width statevector atol=3e-6, policy-selected GPU, and 20q speedup >=1.5x", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"widths": [{"device": "cpu", "error": 1.3119846298259574e-07, "method": "statevector", "mettleq_ms": 4.4152080081403255, "reference_ms": 3.561208985047415, "width": 12}, {"device": "cpu", "error": 1.669086732158931e-07, 

## What should you conclude?

Use the per-width table to choose a crossover on your Mac; do not infer a universal advantage from one width.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.